# SpecForge — Phase 3 Steps E & F on Colab (T4 GPU)

Runs the live retrieval eval (`src/eval/live_run.py`) and the naive baseline
(`src/eval/naive_baseline.py`) on a GPU instance, then downloads the results.

Model: **Qwen/Qwen2.5-3B-Instruct** (natively supported, no `trust_remote_code`).

**Before running:**
1. Prepare `SpecForge.zip` on your machine (see the last cell for the exact command).
2. In Colab: **Runtime → Change runtime type → T4 GPU**, then save.

**Run every cell in order.** The full eval takes ~20–30 minutes.

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU, then restart.")

In [ ]:
# Install only what is missing. Do NOT reinstall torch/transformers here:
# Colab ships a CUDA build of torch, and reinstalling it would break GPU support.
# accelerate is needed for device_map='auto' when the model loads on GPU.
!pip install -q pandas requests beautifulsoup4 numpy faiss-cpu sentence-transformers duckduckgo_search accelerate
print("deps installed")

In [ ]:
import os
import zipfile

zip_path = "/content/SpecForge.zip"

# Only prompt for an upload if the zip is not already present (a previous
# run of this cell saves it to /content/SpecForge.zip). The import is done
# lazily so a missing `files` module can't break this cell when the zip
# already exists.
if not os.path.exists(zip_path):
    print("Upload SpecForge.zip now (a file picker will open)...")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            zip_path = os.path.join("/content", name)
            break
    if not os.path.exists(zip_path):
        raise SystemExit("No zip uploaded — aborting.")

def extract_zip(zip_path, dest):
    """Extract a zip, normalizing backslash separators.

    Windows PowerShell 5.1's Compress-Archive writes entry names with
    backslashes (e.g. ``src\\eval\\live_run.py``). Python's zipfile on
    Linux extracts those as literal filenames, not directories. We rewrite
    every entry to use forward slashes before extracting.
    """
    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            fixed = info.filename.replace("\\", "/")
            target = os.path.join(dest, fixed)
            if info.is_dir():
                os.makedirs(target, exist_ok=True)
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, "wb") as dst:
                dst.write(src.read())
    print("Extracted:", os.path.basename(zip_path))

extract_zip(zip_path, "/content")

# Locate the repo root: the directory that contains src/eval/live_run.py
candidates = ["/content"] + [
    os.path.join("/content", d)
    for d in os.listdir("/content")
    if os.path.isdir(os.path.join("/content", d))
]
repo = next(
    (c for c in candidates if os.path.exists(os.path.join(c, "src", "eval", "live_run.py"))),
    None,
)
if repo is None:
    print("Could not find src/eval/live_run.py. Top-level entries under /content:")
    for d in sorted(os.listdir("/content")):
        print(" -", repr(d))
    raise SystemExit("Check the zip layout above.")

os.chdir(repo)
print("Repo root:", os.getcwd())

In [ ]:
import os

required = [
    "data/raw/input.csv",
    "data/eval/dev_ground_truth.csv",
    "data/lov/materials.json",
    "data/lov/connection_types.json",
    "data/lov/units.json",
    "data/lov/appliance_brands.json",
    "src/eval/live_run.py",
    "src/eval/naive_baseline.py",
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print("MISSING FILES:", missing)
else:
    print("All required files present.")

In [ ]:
# The repo's model.py already auto-loads on GPU when CUDA is available, so no
# patch is needed. Just confirm the model id and GPU handling are present.
from pathlib import Path

text = Path("src/llm/model.py").read_text(encoding="utf-8")
assert 'MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"' in text, "model.py still has the old MODEL_ID"
assert "device_map" in text, "model.py missing GPU device_map handling"
print("model.py OK: Qwen2.5-3B-Instruct + auto-GPU.")

In [ ]:
# Load Qwen2.5-3B-Instruct (downloads ~6 GB on first run) and confirm it is on GPU.
import os
import sys

sys.path.insert(0, os.getcwd())

from src.llm.model import load_llm

tok, mdl = load_llm()
if tok is None or mdl is None:
    raise SystemExit("LLM failed to load.")

print("Tokenizer:", type(tok).__name__)
dev = next(mdl.parameters()).device
print("Model device:", dev)
assert dev.type == "cuda", "Model is NOT on GPU!"
print("Model is on GPU. Ready.")

In [ ]:
# Step C sanity check: 3 extraction cases + 1 generation case.
!python -u scripts/sanity_check_llm.py

In [ ]:
# Smoke test: 2 rows only, to confirm search + fetch + LLM all work end-to-end.
!python -u src/eval/live_run.py --limit 2 --out data/eval/live_run_smoke.json

## Full run (20 rows)

The cell below runs the complete live retrieval eval. Expect **~20–30 minutes**
on a T4: 5 retrieval fields per row, each field = 1 LLM call, plus a DuckDuckGo
search and up to 3 page fetches per row.

Keep this browser tab open while it runs. Progress prints row-by-row.

In [ ]:
!python -u src/eval/live_run.py 2>&1 | tee data/eval/live_run.log

In [ ]:
# Step F: naive baseline (1 LLM call per row, no retrieval). ~2–4 minutes.
!python -u src/eval/naive_baseline.py

In [ ]:
import json

for f in ["data/eval/live_run_results.json", "data/eval/naive_baseline_results.json"]:
    print("=" * 70)
    print(f)
    print("=" * 70)
    with open(f, encoding="utf-8") as fh:
        payload = json.load(fh)
    print(json.dumps(payload.get("summary", {}), indent=2))

In [ ]:
# Download the artifacts back to your machine.
import os
from google.colab import files

for f in [
    "data/eval/live_run_results.json",
    "data/eval/naive_baseline_results.json",
    "data/eval/live_run.log",
    "data/eval/live_run_smoke.json",
]:
    if os.path.exists(f):
        print("Downloading", f)
        files.download(f)
    else:
        print("Skipping (not found):", f)

## How to prepare SpecForge.zip (on your Windows machine)

Open PowerShell in `C:\Users\Rohit\Desktop\SpecForge` and run **one** of these.

**Option A — `tar` (recommended, produces forward-slash entries):**

```powershell
tar -a -c -f "$env:USERPROFILE/Desktop/SpecForge.zip" src data scripts requirements.txt
```

**Option B — PowerShell `Compress-Archive` (works too; the notebook now fixes its
backslash entries automatically):**

```powershell
Compress-Archive -Path "src", "data", "scripts", "requirements.txt" -DestinationPath "$env:USERPROFILE/Desktop/SpecForge.zip"
```

Both include exactly what the notebook needs (source + data + requirements),
excluding `.git` and `__pycache__`. The `data/cache/` HTML files are included
so already-fetched pages are reused instead of re-downloaded.

Then, in Colab, upload `SpecForge.zip` when the upload cell asks for it.

## Troubleshooting

- **No GPU**: Runtime → Change runtime type → T4 GPU, then Runtime → Restart and run all again.
- **DuckDuckGo rate-limited** (search returns nothing → every field is `no_evidence`):
  the pipeline falls back to the HTML endpoint. If Colab's IP is fully blocked,
  wait a bit and re-run, or run the search/fetch step locally and re-upload the
  `data/cache/` folder.
- **Model not on GPU**: make sure the load cell printed "cuda".
- **Model download slow**: Qwen2.5-3B is ~6 GB; first load takes a few minutes.